In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [90]:
import numpy as np
import math
import re
import pandas as pd
import jax.numpy as jnp
from jax.typing import ArrayLike
from scipy.special import factorial

import altair as alt

from scipy.interpolate import BSpline

from mixres.sim import PoissonDisjoint1D

import altair as alt

## Utility Functions

In [87]:
_INTERVAL_PATTERN = re.compile(
    r"^\s*([\[(])\s*([-+]?\d+(?:\.\d+)?)\s*,\s*([-+]?\d+(?:\.\d+)?)\s*([\])])\s*$"
)


def expand_interval_dataframe(
    df: pd.DataFrame,
    interval_col: str | None = None,
    x_col: str = "x",
) -> pd.DataFrame:
    """Expand interval-indexed rows into one row per integer point.

    Parameters
    ----------
    df : pandas.DataFrame
        Input data containing an interval column plus any columns to copy.
    interval_col : str or None, optional
        Name of the interval column. If None, the function will try to infer it
        from string columns containing values like "[0,5)" or "[95, 100]".
    x_col : str, optional
        Name of the integer-valued output column. Defaults to "x".

    Returns
    -------
    pandas.DataFrame
        Expanded dataframe with one row per integer in each interval.
    """
    if df.empty:
        out = df.copy()
        out[x_col] = pd.Series(dtype="int64")
        return out

    working_df = df.copy()

    if interval_col is None:
        candidate_cols = []
        for col in working_df.columns:
            series = working_df[col].dropna().astype(str)
            if not series.empty and series.str.match(_INTERVAL_PATTERN).all():
                candidate_cols.append(col)

        if not candidate_cols:
            raise ValueError(
                "Could not infer the interval column. Pass interval_col explicitly."
            )
        interval_col = candidate_cols[0]

    expanded_rows = []

    for _, row in working_df.iterrows():
        interval_value = row[interval_col]

        if isinstance(interval_value, pd.Interval):
            left = float(interval_value.left)
            right = float(interval_value.right)
            left_closed = interval_value.closed in {"left", "both"}
            right_closed = interval_value.closed in {"right", "both"}
        else:
            match = _INTERVAL_PATTERN.match(str(interval_value))
            if match is None:
                raise ValueError(
                    f"Could not parse interval {interval_value!r} in column {interval_col!r}."
                )

            left_bracket, left_str, right_str, right_bracket = match.groups()
            left = float(left_str)
            right = float(right_str)
            left_closed = left_bracket == "["
            right_closed = right_bracket == "]"

        start = math.ceil(left) if left_closed else math.floor(left) + 1
        end = math.floor(right) if right_closed else math.ceil(right) - 1

        for x in range(start, end + 1):
            new_row = row.to_dict()
            new_row[x_col] = x
            expanded_rows.append(new_row)

    return pd.DataFrame(expanded_rows).reset_index(drop=True)


def make_bspline_basis(
    x: np.ndarray,
    n_basis: int = 30,
    knots: np.ndarray | None = None,
) -> np.ndarray:
    """Construct a cubic B-spline basis matrix for input grid x.

    Parameters
    ----------
    x : np.ndarray, shape (N,)
        Input grid values (1-D).
    n_basis : int, optional
        Number of basis functions (default 30). Ignored when `knots` is given.
    knots : np.ndarray or None, optional
        Interior knot locations. If None, knots are placed uniformly in
        (x.min(), x.max()) such that the basis has `n_basis` functions.
        When supplied, `n_basis` is inferred as len(knots) + degree + 1.

    Returns
    -------
    np.ndarray, shape (N, M)
        Dense basis matrix, where M = n_basis (or len(knots) + degree + 1).
    """
    degree = 3
    x = np.asarray(x, dtype=float)
    x_min, x_max = x.min(), x.max()

    if knots is None:
        n_interior = n_basis - degree - 1
        if n_interior < 0:
            raise ValueError(
                f"n_basis={n_basis} is too small for degree={degree}; "
                f"need n_basis >= {degree + 1}."
            )
        interior_knots = np.linspace(x_min, x_max, n_interior + 2)[1:-1]
    else:
        interior_knots = np.asarray(knots, dtype=float)
        n_basis = len(interior_knots) + degree + 1  # override

    # Clamped knot vector: boundary knots repeated (degree + 1) times
    t = np.concatenate(
        [
            np.repeat(x_min, degree + 1),
            interior_knots,
            np.repeat(x_max, degree + 1),
        ]
    )

    B = BSpline.design_matrix(x, t, degree).toarray()  # shape (N, n_basis)
    return B


def build_interval_sum_matrix(
    intervals: list[str],
    dtype=jnp.float32,
    return_support: bool = False,
):
    """Build a JAX matrix that sums a latent vector over integer intervals.

    Each row corresponds to one interval string such as "[0,5)" or "[95, 100]".
    The columns correspond to the integer support from the smallest included
    integer to the largest included integer. Entries are 1 inside the interval
    and 0 elsewhere.
    """
    if not intervals:
        raise ValueError("intervals must contain at least one interval string.")

    parsed_intervals = []
    min_x = np.inf
    max_x = -np.inf

    for interval in intervals:
        match = _INTERVAL_PATTERN.match(str(interval))
        if match is None:
            raise ValueError(f"Could not parse interval: {interval!r}")

        left_bracket, left_str, right_str, right_bracket = match.groups()
        left = float(left_str)
        right = float(right_str)

        start = math.ceil(left) if left_bracket == "[" else math.floor(left) + 1
        end = math.floor(right) if right_bracket == "]" else math.ceil(right) - 1

        if end < start:
            raise ValueError(f"Interval contains no integer points: {interval!r}")

        parsed_intervals.append((start, end))
        min_x = min(min_x, start)
        max_x = max(max_x, end)

    width = int(max_x - min_x + 1)
    matrix = np.zeros((len(parsed_intervals), width), dtype=np.float32)

    for row_idx, (start, end) in enumerate(parsed_intervals):
        matrix[row_idx, int(start - min_x) : int(end - min_x + 1)] = 1.0

    matrix = jnp.asarray(matrix, dtype=dtype)

    if return_support:
        support = jnp.arange(int(min_x), int(max_x) + 1)
        return matrix, support

    return matrix


def diff_matrix(num_nodes: int, order: int) -> ArrayLike:
    """
    Construct a finite difference matrix of given order.

    Parameters
    ----------
    num_nodes: int
        The number of nodes in the grid.
    order: int
        The order of the finite difference.

    Returns
    -------
    D: ArrayLike
        The finite difference matrix of shape (num_nodes - order, num_nodes).
    """
    D = jnp.zeros((num_nodes - order, num_nodes))
    i_vals = jnp.arange(order + 1)
    coeff = (factorial(order) / (factorial(i_vals) * factorial(order - i_vals))) * (
        -1
    ) ** (order - i_vals)
    for i in range(num_nodes - order):
        D = D.at[i, i : i + order + 1].set(coeff)

    return D

## Analysis

In [102]:
# Generate the dataset
df = PoissonDisjoint1D(interval_width=20, n_observations=20, seed=42).generate()

# Visualize the dataset
df_expanded = expand_interval_dataframe(df, interval_col="interval", x_col="x")
base = alt.Chart(df_expanded).mark_point(size=2).encode(x="x", y="y")
true_line = alt.Chart(df_expanded).mark_line(color="red").encode(x="x", y="lambda")

base + true_line

alt.LayerChart(...)

## Modeling

In [103]:
df_model = df.groupby(["interval_id"])["y"].agg(N="size", y="sum").reset_index()

In [104]:
x = np.arange(0, 101)
y_obs = df_model["y"].values
interval_id = df_model["interval_id"].values
N = df_model["N"].values
A = build_interval_sum_matrix(df["interval"].unique().tolist())
B = make_bspline_basis(x, n_basis=40)

# Convert to JAX arrays
y_obs = jnp.array(y_obs)
interval_id = jnp.array(interval_id)
N = jnp.array(N)
A_jax = jnp.array(A)
B_jax = jnp.array(B)

In [105]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import autoguide

from mixres.models._inference import (
    run_inference_mcmc,
    run_inference_svi,
    posterior_predictive_mcmc,
    posterior_predictive_svi,
)

In [106]:
def model_bspline(
    A: ArrayLike, B: ArrayLike, N: ArrayLike, interval_id: ArrayLike, y_obs=None
):
    """
    Bayesian B-spline regression model.

    Priors:
        beta0  ~ N(0, 1)            global intercept
        w      ~ N(0, I_K)          B-spline coefficients

    Likelihood:
        y | beta0, w ~ Poisson(exp(beta0 + B @ w))
    """
    K = B.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + B @ w))
    lam_interval = (
        jnp.matmul(A, lam) / 20 * N[interval_id]
    )  # Average rate over each interval (divide by interval width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y_obs)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [107]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_bspline,
    A=A_jax,
    B=B_jax,
    N=N,
    interval_id=interval_id,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 5061.13it/s, 15 steps of size 2.58e-01. acc. prob=0.92]


Number of divergences: 0
dict_keys(['beta0', 'lam', 'w'])


In [108]:
# Extract posterior mean of mu from MCMC samples
lam = mcmc_samples["lam"].mean(axis=0)
lam_lower = jnp.percentile(mcmc_samples["lam"], 2.5, axis=0)
lam_upper = jnp.percentile(mcmc_samples["lam"], 97.5, axis=0)

df_results = df_expanded[["x", "lambda"]].drop_duplicates().reset_index(drop=True)
df_results["lam"] = lam
df_results["lam_lower"] = lam_lower
df_results["lam_upper"] = lam_upper

# Visualize results
true_line = alt.Chart(df_results).mark_line(color="red").encode(x="x", y="lambda")
mean_line = alt.Chart(df_results).mark_line(color="blue").encode(x="x", y="lam")
bands = (
    alt.Chart(df_results)
    .mark_area(color="lightblue", opacity=0.5)
    .encode(
        x="x",
        y="lam_lower:Q",
        y2="lam_upper:Q",
    )
)
true_line + mean_line + bands

alt.LayerChart(...)

In [109]:
def model_pspline(
    A: ArrayLike,
    B: ArrayLike,
    D: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    y_obs: ArrayLike = None,
    epsilon: float = 1e-3,
):
    """
    Bayesian P-spline regression model.

    Parameters
    ----------
    B: ArrayLike
        B-spline basis matrix of shape (N, K).
    D: ArrayLike
        Finite difference matrix of shape (K - order, K) for the penalty.
    y_obs: ArrayLike, optional
        Observed data of shape (N,). If None, the model is treated as a generative model.
    epsilon: float, optional
                Soft constraint strength (default 1e-3). Smaller values enforce smoother solutions.

    Priors:
        beta0  ~ N(0, 1)            global intercept
        sigma2 ~ Gamma(2, 1)        observation noise variance
        w      ~ N(0, I_K)          B-spline coefficients

    Likelihood:
        y | beta0, w, sigma2 ~ N(beta0 + B @ w, sigma2)
    """
    K = B.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))

    # Prior on the differences
    delta = numpyro.sample("delta", dist.Normal(0, 1).expand([D.shape[0]]).to_event(1))
    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # 4. The Soft Constraints (Vectorized)
    diff = jnp.matmul(D, w)
    penalty = -0.5 * jnp.sum(jnp.square(diff - delta)) / (epsilon**2)
    numpyro.factor("soft_constraint", penalty)

    # --- Latent mean ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + B @ w))
    lam_interval = (
        jnp.matmul(A, lam) / 20 * N[interval_id]
    )  # Average rate over each interval (divide by interval width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y_obs)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [110]:
D_jax = jnp.array(diff_matrix(B.shape[1], order=2))

In [111]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_pspline,
    A=A_jax,
    B=B_jax,
    D=D_jax,
    N=N,
    interval_id=interval_id,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:04<00:00, 213.55it/s, 1023 steps of size 6.00e-04. acc. prob=0.86]

Number of divergences: 0
dict_keys(['beta0', 'delta', 'lam', 'w'])


In [112]:
# Extract posterior mean of mu from MCMC samples
lam = mcmc_samples["lam"].mean(axis=0)
lam_lower = jnp.percentile(mcmc_samples["lam"], 2.5, axis=0)
lam_upper = jnp.percentile(mcmc_samples["lam"], 97.5, axis=0)

df_results = df_expanded[["x", "lambda"]].drop_duplicates().reset_index(drop=True)
df_results["lam"] = lam
df_results["lam_lower"] = lam_lower
df_results["lam_upper"] = lam_upper

# Visualize results
true_line = alt.Chart(df_results).mark_line(color="red").encode(x="x", y="lambda")
mean_line = alt.Chart(df_results).mark_line(color="blue").encode(x="x", y="lam")
bands = (
    alt.Chart(df_results)
    .mark_area(color="lightblue", opacity=0.5)
    .encode(
        x="x",
        y="lam_lower:Q",
        y2="lam_upper:Q",
    )
)
true_line + mean_line + bands

alt.LayerChart(...)

In [113]:
def model_fHS(
    A: ArrayLike,
    B: ArrayLike,
    D: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    y_obs: ArrayLike = None,
    epsilon: float = 1e-3,
):
    """
    Bayesian P-spline regression model.

    Parameters
    ----------
    B: ArrayLike
        B-spline basis matrix of shape (N, K).
    D: ArrayLike
        Finite difference matrix of shape (K - order, K) for the penalty.
    y_obs: ArrayLike, optional
        Observed data of shape (N,). If None, the model is treated as a generative model.
    epsilon: float, optional
                Soft constraint strength (default 1e-3). Smaller values enforce smoother solutions.

    Priors:
        beta0  ~ N(0, 1)            global intercept
        sigma2 ~ Gamma(2, 1)        observation noise variance
        w      ~ N(0, I_K)          B-spline coefficients

    Likelihood:
        y | beta0, w, sigma2 ~ N(beta0 + B @ w, sigma2)
    """
    K = B.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))

    # Prior on the differences
    tau = numpyro.sample("tau", dist.HalfCauchy(1.0))
    _lambda = numpyro.sample("_lambda", dist.HalfCauchy(1.0).expand([D.shape[0]]))
    delta = numpyro.sample("delta", dist.Normal(0, 1).expand([D.shape[0]]).to_event(1))
    delta = delta * tau**2 * _lambda**2

    w = numpyro.sample("w", dist.Normal(0, 1).expand([K]).to_event(1))

    # 4. The Soft Constraints (Vectorized)
    diff = jnp.matmul(D, w)
    penalty = -0.5 * jnp.sum(jnp.square(diff - delta)) / (epsilon**2)
    numpyro.factor("soft_constraint", penalty)

    # --- Latent mean ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + B @ w))
    lam_interval = (
        jnp.matmul(A, lam) / 20 * N[interval_id]
    )  # Average rate over each interval (divide by interval width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y_obs)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y_obs)

In [114]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_fHS,
    A=A_jax,
    B=B_jax,
    D=D_jax,
    N=N,
    interval_id=interval_id,
    y_obs=y_obs,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:06<00:00, 166.30it/s, 1023 steps of size 1.31e-03. acc. prob=0.88]

Number of divergences: 139
dict_keys(['_lambda', 'beta0', 'delta', 'lam', 'tau', 'w'])


In [115]:
# Extract posterior mean of mu from MCMC samples
lam = mcmc_samples["lam"].mean(axis=0)
lam_lower = jnp.percentile(mcmc_samples["lam"], 2.5, axis=0)
lam_upper = jnp.percentile(mcmc_samples["lam"], 97.5, axis=0)

df_results = df_expanded[["x", "lambda"]].drop_duplicates().reset_index(drop=True)
df_results["lam"] = lam
df_results["lam_lower"] = lam_lower
df_results["lam_upper"] = lam_upper

# Visualize results
true_line = alt.Chart(df_results).mark_line(color="red").encode(x="x", y="lambda")
mean_line = alt.Chart(df_results).mark_line(color="blue").encode(x="x", y="lam")
bands = (
    alt.Chart(df_results)
    .mark_area(color="lightblue", opacity=0.5)
    .encode(
        x="x",
        y="lam_lower:Q",
        y2="lam_upper:Q",
    )
)
true_line + mean_line + bands

alt.LayerChart(...)